# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
# DEBIAN_FRONTEND=noninteractive so any apt post-install prompt
# doesn't stall the whole boot silently (fontconfig sometimes prompts
# on a fresh Kaggle image).
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.check_call(['apt-get', '-qq', 'update'])
# fonts-noto{,-cjk,-color-emoji} + fonts-dejavu-core — needed for
# burned-in captions to render non-Latin scripts (Devanagari, Bengali,
# CJK, Arabic, Thai). DejaVu Sans is the primary font (bundled default)
# with libass falling back to Noto CJK for CJK. -y and DEBIAN_FRONTEND
# ensure no interactive prompt.
subprocess.check_call(['apt-get', '-qq', 'install', '-y',
                       '--no-install-recommends',
                       'ffmpeg', 'fonts-dejavu-core',
                       'fonts-noto', 'fonts-noto-cjk',
                       'fonts-noto-color-emoji'])
subprocess.run(['fc-cache', '-f'], check=False)

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — batched where safe, split when a per-package flag
#    would otherwise leak to every package in the same pip call.
import subprocess, sys, os

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'espeak-ng'])
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'phonemizer'], check=False)

# MAIN BATCH — no per-package flags.
# transformers CAPPED at <4.50 — anything newer removes AlbertModel
# (Kokoro) + PreTrainedModel (diffusers <0.32).
# diffusers CAPPED at <0.32 — 0.32+ ships flux2 pipeline that lazy-
# imports Qwen3ForCausalLM (needs transformers >=4.50). Two-way pin
# conflict without both caps. Confirmed live 2026-07-09.
# pyOpenSSL >= 25 + cryptography >= 45 avoid the GEN_EMAIL crash.
#
# NOTE (2026-07-11): The klein-4B local provider (added to shotfinder.py
# in ea90851) requires diffusers>=0.36 + transformers>=4.51, but those
# break Kokoro's transformers<4.50 cap. We tried it — Kaggle boot failed
# in this cell within 2-4 min. Reverted here to the last-known-good
# pins; klein-4B provider stays in shotfinder.py but auto-disables at
# runtime because `from diffusers import Flux2KleinPipeline` will
# ImportError → _LOCAL_FLUX2_BROKEN=True → provider skipped in the
# chain, SDXL keeps serving as before. To re-enable klein-4B later,
# figure out the specific resolver conflict from a Kaggle web-UI log
# and bump both pins together with a compatible Kokoro release.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '-r', 'requirements.txt',
    '-r', 'requirements-browser.txt',
    'decord==0.6.0', 'av==12.3.0',
    'hf_transfer',
    'espeakng-loader>=0.2',
    'soundfile>=0.12',
    'kokoro>=0.9.4', 'misaki>=0.9.4',
    'transformers>=4.44,<4.50', 'accelerate>=0.30', 'safetensors>=0.4',
    'huggingface_hub>=0.23', 'peft>=0.10',
    'pyOpenSSL>=25.0.0', 'cryptography>=45.0.0',
])

# SEPARATE — per-package flags kept out of the main batch so they
# don't leak to every dep. Each in its own subprocess.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--force-reinstall', 'phonemizer-fork>=3.3.2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--upgrade', 'edge-tts>=7.0.0'])

# SEPARATE — diffusers with --no-deps + version cap. Protects the
# preinstalled CUDA torch AND avoids the flux2 pipeline that requires
# transformers>=4.50.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'diffusers>=0.30,<0.32',
])

# Chromium install in the background — cell exits without waiting.
_playwright_log = open('/tmp/playwright-install.log', 'w')
_playwright_bg = subprocess.Popen(
    [sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
    stdout=_playwright_log, stderr=subprocess.STDOUT,
)
print(f'playwright chromium install running in background (pid={_playwright_bg.pid})', flush=True)

import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
for _mod, _label in [('torchvision','torchvision'),('transformers','transformers'),('diffusers','diffusers')]:
    try:
        _m = __import__(_mod)
        print(f'{_label}:', _m.__version__)
    except Exception as e:
        print(f'{_label}: NOT ready ->', e)
try:
    from kokoro import KPipeline
    print('kokoro: ready')
except Exception as e:
    print('kokoro: NOT ready ->', e)
try:
    from diffusers import AutoPipelineForText2Image  # noqa: F401
    print('diffusers AutoPipeline: ready (local_sdxl provider available)')
except Exception as e:
    print('diffusers AutoPipeline: NOT ready ->', e)
# Klein-4B check — expected to fail with old diffusers pin. Not fatal.
try:
    from diffusers import Flux2KleinPipeline  # noqa: F401
    print('diffusers Flux2KleinPipeline: ready (local_flux2_klein provider available)')
except Exception as e:
    print('diffusers Flux2KleinPipeline: not available with pinned diffusers<0.32 '
          '(expected — klein-4B provider will auto-skip; chain uses SDXL fallback)')
try:
    import edge_tts
    print('edge-tts:', getattr(edge_tts, '__version__', 'installed'))
except Exception as e:
    print('edge-tts: NOT ready ->', e)
try:
    import OpenSSL, cryptography
    print(f'crypto: pyOpenSSL={OpenSSL.__version__} cryptography={cryptography.__version__}')
except Exception as e:
    print('crypto: NOT ready ->', e)

In [ ]:
# 3) BOOTSTRAP secrets.
#
# Two sources, in priority order:
#
#   (1) A private Kaggle Dataset attached via kernel-metadata.json. This
#       is the production path — survives `kaggle kernels push` cycles
#       (each push creates a new kernel version, but the dataset stays
#       attached). Create one private Dataset named e.g.
#       "yt-agent-secrets" with ONE file:
#
#           secrets.env
#               COOLIFY_BASE_URL=https://yt-agent.thyker.online
#               PB_URL=https://yt-agent.thyker.online/pb
#               POCKETBASE_ADMIN_EMAIL=admin@yt-agent.thyker.online
#               POCKETBASE_ADMIN_PASSWORD=your-password
#               RENDER_TRIGGER_KEY=...
#               STORAGE_PROVIDERS_ENC_KEY=...
#
#       Then reference it in kernel-metadata.json:
#           "dataset_sources": ["<your-username>/yt-agent-secrets"]
#
#   (2) Kaggle's Secrets panel (Add-ons → Secrets). Useful for
#       interactive testing. Same key names as the .env file.
#
# Either source works — the notebook reads whichever it finds first.
import os, glob

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    # Legacy Vercel + Firestore path — keep working for users who
    # haven\'t migrated to Coolify yet.
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]
ALL_KEYS = REQUIRED + OPTIONAL


def _load_from_dataset_envfile():
    """Look for an attached Dataset secrets file. Kaggle mounts datasets
    under /kaggle/input/<dataset-name>/. Search for the most common
    file names."""
    candidates = []
    for root in glob.glob('/kaggle/input/*/'):
        for name in ('secrets.env', '.env', 'env.txt', 'secrets.txt'):
            p = os.path.join(root, name)
            if os.path.exists(p):
                candidates.append(p)
    if not candidates:
        return False
    found = False
    for path in candidates:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k in ALL_KEYS and v:
                    os.environ.setdefault(k, v)
                    found = True
        print(f'  loaded secrets from {path}')
    return found


def _load_from_kaggle_secrets():
    """Read from the Kaggle Secrets panel. Fails silently when not
    available (e.g. running in a Colab clone of the notebook)."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        loaded = 0
        for k in ALL_KEYS:
            try:
                v = secrets.get_secret(k)
            except Exception:
                continue
            if v and not os.environ.get(k):
                os.environ[k] = v
                loaded += 1
        if loaded:
            print(f'  loaded {loaded} secrets from Kaggle Secrets panel')
        return loaded > 0
    except Exception as e:
        print(f'  Kaggle Secrets unavailable ({e!r}); using Dataset only')
        return False


# Try both sources — Dataset first, then Kaggle Secrets fills any gaps.
got_dataset = _load_from_dataset_envfile()
got_secrets = _load_from_kaggle_secrets()
if not (got_dataset or got_secrets):
    raise SystemExit(
        'No secrets source found. Either attach the yt-agent-secrets '
        'Dataset (via kernel-metadata.json dataset_sources) OR add the '
        'required keys to the Kaggle Secrets panel.'
    )

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise SystemExit('MISSING REQUIRED SECRETS: ' + ', '.join(missing))

# Defaults for the new outbound-poll mode.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'kaggle-gpu')
# Idle auto-shutdown ~25 min — preserves the 30 GPU-hr/week budget.
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

print('Kaggle bootstrap OK')
print('  Dashboard:', os.environ.get('COOLIFY_BASE_URL'))
print('  Pocketbase:', os.environ.get('PB_URL'))
print('  Mode:', os.environ.get('WORKER_MODE'), '| tier:', os.environ.get('INSTANCE_TIER'))


In [ ]:
# 4) (Skipped in outbound-poll mode.) Legacy cloudflared tunnel cell.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 4.5) Pre-warm SDXL weights in the BACKGROUND — doesn't block boot.
#
# Only downloads the fp16 variants. Our shotfinder loads with
# variant="fp16" so the fp32 .safetensors files (unet 10.3G, text
# encoders 3G+, top-level bundle 13.9G) are pure waste. Narrow the
# allow_patterns to fp16-only + config files → ~7 GB instead of ~35 GB.
#
# Klein-4B snapshot was here temporarily (see git log ea90851 for the
# planned config) but the Kaggle dep upgrade broke boot. Klein-4B is
# disabled in shotfinder via ImportError → chain uses SDXL as before.
# When we retry klein-4B with a real Kaggle log to diagnose the
# resolver conflict, re-add the klein-4B snapshot as a second
# daemon thread alongside SDXL.
import os, time, subprocess, threading
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '180')

def _bg_snapshot_sdxl():
    try:
        t0 = time.time()
        print('preboot(bg): snapshotting SDXL fp16 weights into HF cache...', flush=True)
        from huggingface_hub import snapshot_download
        _sdxl_model = os.environ.get('LOCAL_SDXL_MODEL', 'stabilityai/sdxl-turbo')
        snapshot_download(
            _sdxl_model,
            allow_patterns=[
                # Configs + tokenizer json/text files.
                '*.json', '*.txt',
                # fp16 model weights only (skips ~28 GB of fp32 dupes).
                '**/*.fp16.safetensors',
                'sd_xl_turbo_1.0_fp16.safetensors',
            ],
        )
        print(f'preboot(bg): SDXL fp16 weights cached in {time.time()-t0:.1f}s', flush=True)
    except Exception as e:
        print(f'preboot(bg): SDXL cache warm skipped ({e!r})', flush=True)

threading.Thread(target=_bg_snapshot_sdxl, name='sdxl-snapshot', daemon=True).start()
print('preboot: SDXL snapshot kicked in background thread', flush=True)

try:
    out = subprocess.check_output(['nvidia-smi', '-L'], timeout=5).decode().strip()
    print('preboot: nvidia-smi -L:\n' + out, flush=True)
    if out.count('GPU ') < 2:
        print('preboot: WARNING - only 1 GPU visible. To use T4x2, set '
              'Notebook Settings -> Accelerator -> GPU T4 x2 -> Save Version.',
              flush=True)
except Exception as e:
    print(f'preboot: nvidia-smi -L failed ({e!r})', flush=True)

print('preboot: done (main thread not blocked on SDXL)', flush=True)

In [ ]:
# 4.6) Self-check — verify multi-GPU + multilingual wiring
#
# Wrapped in try/except so any check crash still lets uvicorn boot.
# The self-check is diagnostic, not a gate.
try:
    from backend import self_check
    print(self_check.run(text=True), flush=True)
except Exception as e:
    print(f'self_check skipped ({e!r})', flush=True)

In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])